In [1]:
import os
from typing import TypedDict

import psycopg
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

load_dotenv()
DATABASE_URL = os.environ["DATABASE_URL"]
CURRENT_CUSTOMER_ID = 101

llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)
print("Model configured")

Model configured


In [2]:
def get_connection():
    return psycopg.connect(DATABASE_URL)

ORDER_COLUMNS = "order_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at"

def order_from_row(row):
    minutes_late = None
    if row[6] and row[7]:
        minutes_late = round((row[7] - row[6]).total_seconds() / 60)
    return {
        "order_id": row[0], "restaurant": row[1], "items": row[2], "status": row[3],
        "total_amount": float(row[4]), "created_at": str(row[5]),
        "promised_at": str(row[6]) if row[6] else None,
        "delivered_at": str(row[7]) if row[7] else None,
        "minutes_late": minutes_late,
    }

def query_latest_order(customer_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE customer_id=%s ORDER BY created_at DESC LIMIT 1",
            (customer_id,),
        ).fetchone()

def query_order(customer_id, order_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE order_id=%s AND customer_id=%s",
            (order_id, customer_id),
        ).fetchone()

In [3]:

@tool
def basic_get_latest_order() -> dict:
    """Get the latest order for the current classroom customer."""
    row = query_latest_order(CURRENT_CUSTOMER_ID)
    if row is None:
        return {"found": False, "message": "No orders were found."}
    return {"found": True, "order": order_from_row(row)}

@tool
def basic_get_order_details(order_id: str) -> dict:
    """Get details for an order ID belonging to the current classroom customer."""
    row = query_order(CURRENT_CUSTOMER_ID, order_id)
    if row is None:
        return {"found": False, "message": "No matching order was found."}
    return {"found": True, "order": order_from_row(row)}

### LangChain Agent

In [4]:
BASIC_SYSTEM_PROMPT = """You are a Zomato customer-support assistant.
Use tools to verify order information. Never invent order details.
If a tool says an order was not found, explain that you could not verify it.
Be concise and do not claim to have performed actions you did not perform."""

basic_tools = [basic_get_latest_order, basic_get_order_details]

basic_agent = create_agent(
    model=llm,
    tools=basic_tools,
    system_prompt=BASIC_SYSTEM_PROMPT,
)
print("Agent tools:", [t.name for t in basic_tools])

Agent tools: ['basic_get_latest_order', 'basic_get_order_details']


In [5]:
def ask_basic(question):
    result = basic_agent.invoke({
        "messages": 
            [{"role": "user", "content": question}]
        }
    )
    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            print("MODEL REQUESTED:", [(call["name"], call["args"]) for call in message.tool_calls])
        elif message.type == "tool":
            print("TOOL RESULT:", str(message.content)[:400])
    print("FINAL:", result["messages"][-1].content)
    return result


ask_basic("Where is my latest order?")

MODEL REQUESTED: [('basic_get_latest_order', {})]
TOOL RESULT: {"found": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-12 17:48:25.940790+00:00", "promised_at": "2026-11-12 18:28:25.940790+00:00", "delivered_at": null, "minutes_late": null}}
FINAL: Your latest order is from **Burger Barn**. Here are the details:

- **Order ID:** ORD-1003
- **Items:** Veg Burger x2; Fries x1
- **Status:** Out for Delivery
- **Total Amount:** ₹350
- **Order Created At:** November 12, 2026, 17:48
- **Promised Delivery Time:** November 12, 2026, 18:28

Your order is currently on its way!


{'messages': [HumanMessage(content='Where is my latest order?', additional_kwargs={}, response_metadata={}, id='89b278e8-3a60-45b4-a204-c81242312cc5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 134, 'total_tokens': 146, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 2.73e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.73e-05, 'upstream_inference_prompt_cost': 2.01e-05, 'upstream_inference_completions_cost': 7.2e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'gen-1794514524-Kmbwbse5fSsqoGI3gCJL', 'service_tier'

In [8]:
ask_basic("Can you tell me the details for my order with ID ORD-2001?")

MODEL REQUESTED: [('basic_get_order_details', {'order_id': 'ORD-2001'})]
TOOL RESULT: {"found": false, "message": "No matching order was found."}
FINAL: I could not verify the order with ID ORD-2001, as no matching order was found.


{'messages': [HumanMessage(content='Can you tell me the details for my order with ID ORD-2001?', additional_kwargs={}, response_metadata={}, id='88d666ea-7040-42d9-9988-dc2eca229ed0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 144, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 3.36e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.36e-05, 'upstream_inference_prompt_cost': 2.16e-05, 'upstream_inference_completions_cost': 1.2e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'gen-1794514805-nEBx